# Rigid multi-pose public continuity receiving

Prospective qualification only. Execute only under a separately registered supervisor.
Receive all N=2..3 poses of the newly generated rigid result; preserve the earlier incomplete N2 and failed N1 populations.
The public three-cell example remains unchanged. No H generation or search is authorized here; new fixed scores are explicitly counted.


In [ ]:
import base64
import gzip
import hashlib
import json
import os
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

root = Path(os.environ['BINDINGMT_CONTINUITY_ROOT']).resolve()
generation = Path(os.environ['BINDINGMT_CONTINUITY_GENERATION']).resolve()
output = Path(os.environ['BINDINGMT_CONTINUITY_OUTPUT']).resolve()
assert not output.exists()
assert not root.is_relative_to(output) and not generation.is_relative_to(output)
assert not output.is_relative_to(generation)
assert os.sched_getaffinity(0) == {0}
assert os.getpriority(os.PRIO_PROCESS, 0) >= 10
output.mkdir(parents=True, exist_ok=False)


def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def relative_name(value):
    path = PurePosixPath(value)
    assert not path.is_absolute() and '..' not in path.parts
    assert path.parts and str(path) == value
    return value


def verify_manifest(directory, expected):
    manifest_path = directory / 'manifest.json'
    assert file_sha(manifest_path) == expected
    manifest = json.loads(manifest_path.read_text())
    assert manifest['owner'] == 'uibcdf/bindingmt#78'
    members = {}
    for row in manifest['files']:
        name = relative_name(row['path'])
        path = directory / name
        assert path.resolve().is_relative_to(directory)
        assert name not in members
        stored = path.read_bytes()
        assert len(stored) == row['bytes']
        assert hashlib.sha256(stored).hexdigest() == row['sha256']
        stored_inputs[str(path)] = row['sha256']
        if name.endswith('.gz'):
            decoded = gzip.decompress(stored)
            assert len(decoded) == row['decoded_bytes']
            assert hashlib.sha256(decoded).hexdigest() == row['decoded_sha256']
        logical = relative_name(row.get('original_path', name))
        assert logical not in members
        members[logical] = row
    stored_inputs[str(manifest_path)] = expected
    return members


def member_bytes(directory, members, logical):
    matches = [
        row
        for name, row in members.items()
        if name == logical or row['path'] == logical
    ]
    assert len(matches) == 1, logical
    row = matches[0]
    stored = (directory / row['path']).read_bytes()
    assert hashlib.sha256(stored).hexdigest() == row['sha256']
    return gzip.decompress(stored) if row['path'].endswith('.gz') else stored


stored_inputs = {}
generation_sha = os.environ['BINDINGMT_CONTINUITY_GENERATION_SHA256']
generation_members = verify_manifest(generation, generation_sha)
source_wire_bytes = member_bytes(generation, generation_members, 'source_result.json')
source_wire_record = json.loads(source_wire_bytes)
generated_inspection = json.loads(
    member_bytes(generation, generation_members, 'source_prepared_inspection.json')
)
population_size = len(source_wire_record['poses'])
assert 2 <= population_size <= 3, 'Receive every actual generated pose, N=2..3.'
assert all(
    set(pose['scores']) == {'vina', 'inter', 'intra', 'torsion'}
    for pose in source_wire_record['poses']
)

construction = root / 'devguide/validation/data/e1_complete_pose_2026-11-06'
construction_sha = '45b3665ba09b6979aa1bcdf1bea0ac3e8102f081fefbfc51348c55bc71405df3'
construction_members = verify_manifest(construction, construction_sha)
full18_name = 'normal/pose_1/full18_chemical_only.h5msm'
full18_row = next(
    row
    for name, row in construction_members.items()
    if name == full18_name or row['path'] == full18_name + '.gz'
)
assert (
    full18_row['sha256']
    == 'bc81efd6a6d6de00141fb03473970a28f356c54bf95eefdf769fd972229bc468'
)
contract = json.loads(
    member_bytes(
        construction, construction_members, 'prospective_scoring_contract.json'
    )
)
inputs = output / 'inputs'
inputs.mkdir()
source_path = inputs / 'full18.h5msm'
source_path.write_bytes(member_bytes(construction, construction_members, full18_name))
receptor_path = inputs / 'receptor.pdbqt'
receptor_path.write_bytes(
    member_bytes(construction, construction_members, 'provided_receptor.pdbqt')
)
assert file_sha(receptor_path) == contract['receptor_sha256']
local_input_hashes = {str(p): file_sha(p) for p in (source_path, receptor_path)}
(output / 'source_result_wire.json').write_bytes(source_wire_bytes)
(output / 'generation_source_prepared_inspection.json').write_text(
    json.dumps(generated_inspection, indent=2, allow_nan=False) + '\n'
)
user_notebook = Path(os.environ['BINDINGMT_CONTINUITY_USER_NOTEBOOK']).resolve()
user_notebook_sha = '41536ca9382e89ec83dca5e3c7a34a8e98be6fab7dd282b85af34b6139184b50'
assert file_sha(user_notebook) == user_notebook_sha
user_cells = [
    cell
    for cell in json.loads(user_notebook.read_text())['cells']
    if cell['cell_type'] == 'code'
]
assert len(user_cells) == 3
stored_inputs[str(user_notebook)] = user_notebook_sha


def event(stage, **fields):
    with (output / 'operations.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    stage=stage, at_utc=datetime.now(timezone.utc).isoformat(), **fields
                ),
                allow_nan=False,
            )
            + '\n'
        )


event(
    'trusted-inputs',
    generation_manifest_sha256=generation_sha,
    actual_poses=population_size,
    user_notebook_sha256=user_notebook_sha,
)

In [ ]:
from functools import wraps
from types import SimpleNamespace

import molsysmt as msm
import numpy as np
import pyunitwizard as puw
import vina
from smonitor.integrations import CatalogException

import bindingmt as dmt
import bindingmt.comparison as comparison_api
from bmt_test_support.qualify_chemical_templates import snapshot

installed_check = globals()['registered_installed_check']
installed_check('before-continuity-receiving')
assert vina.__version__ == '1.2.7'


def plain(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if puw.is_quantity(value):
        return puw.convert(value, to_form='record').to_dict()
    raise TypeError(type(value).__name__)


def normalized(value):
    return json.loads(json.dumps(value, default=plain, allow_nan=False))


def save(path, record):
    with path.open('x') as stream:
        json.dump(normalized(record), stream, indent=2, allow_nan=False)
        stream.write('\n')


def inspect_native(system, mechanics=True):
    result = dict(
        chemistry=snapshot(system),
        ids=[str(x) for x in msm.get(system, element='atom', atom_id=True)],
        names=[str(x) for x in msm.get(system, element='atom', name=True)],
        elements=[
            str(x)
            for x in msm.get(
                system,
                element='atom',
                atom_type=True,
                chemical_state='structure',
                structure_indices=0,
            )
        ],
        coordinates_nm=puw.get_value(
            msm.get(system, element='atom', coordinates=True),
            to_unit='nm',
            value_type=list,
            dtype=float,
        ),
    )
    if mechanics:
        raw_charge = msm.get(system, element='atom', partial_charge=True)
        charge = (
            raw_charge
            if puw.is_quantity(raw_charge)
            else puw.quantity(raw_charge, 'elementary_charge')
        )
        result.update(
            charges_e=puw.get_value(
                charge, to_unit='elementary_charge', value_type=list, dtype=float
            ),
            types=list(msm.get(system, element='atom', atom_ff_type=True)),
            charge_assignment=system.molecular_mechanics.partial_charge_assignment,
            type_assignment=system.molecular_mechanics.atom_type_assignment,
        )
    return normalized(result)


def inspect_prepared(prepared):
    return normalized(
        dict(
            declaration=prepared.to_dict(),
            coordinates_nm=puw.get_value(
                prepared.coordinates, to_unit='nm', value_type=list, dtype=float
            ),
            native=inspect_native(prepared.source_molsys),
        )
    )


# Read the original record before changing the caller's unit policy. Preserve
# its raw wire bytes separately; quantity normalization does not rewrite them.
source_result = dmt.DockingResult.from_dict(deepcopy(source_wire_record))
source_record = source_result.to_dict()
for original, canonical in zip(source_wire_record['poses'], source_record['poses']):
    assert {k: v for k, v in original.items() if k != 'coordinates'} == {
        k: v for k, v in canonical.items() if k != 'coordinates'
    }
    xyz = original['coordinates']
    np.testing.assert_allclose(
        puw.get_value(puw.quantity(xyz['value'], xyz['unit']), to_unit='nm'),
        canonical['coordinates']['value'],
        rtol=0,
        atol=1e-12,
    )
assert len(source_result) == population_size
assert [pose.rank for pose in source_result] == list(range(1, population_size + 1))
assert {pose.partner_state_id for pose in source_result} == {
    'e1-rigid-continuity-benzamidinium-20261107'
}
assert {pose.receptor_state_id for pose in source_result} == {
    'e1-captured-3PTB-receptor-20261020'
}
assert len({pose.pose_id for pose in source_result}) == population_size
assert source_result.ranking_history
assert any(row.get('origin') == 'backend' for row in source_result.ranking_history)
save(output / 'source_result.json', source_record)
source_audit = dmt.audit_result(source_record)
save(output / 'source_audit.json', source_audit)
assert source_audit['status'] == 'consistent'
dmt.verify_captured_inputs(source_record['provenance']['backend_artifacts'])

budgets = dict(
    prepare=population_size + 1,
    reconstruct=population_size,
    source_charge_assignment=1,
    source_type_assignment=1,
    score_attempts=2 * population_size + 2,
    score_returns=2 * population_size,
    graph_witness=4 * population_size + 1,
    charge_audit=3 * population_size + 1,
    comparison_attempts=2,
    comparison_returns=2,
    expected_score_refusals=2,
    rank_refusals=2,
    native_initializations=2 * population_size,
    native_maps=2 * population_size,
    native_scores=2 * population_size,
)
counts = dict.fromkeys(budgets, 0)
patches, preparations, reconstructions, returned_scores = [], [], [], []
forbidden_attempts = []


def consume(name):
    assert counts[name] < budgets[name], (name, counts[name], budgets[name])
    counts[name] += 1


def patch(owner, name, replacement):
    original = getattr(owner, name)
    patches.append((owner, name, original))
    setattr(owner, name, replacement)
    return original


def forbidden(name):
    def reject(*args, **kwargs):
        forbidden_attempts.append(name)
        event('forbidden-operation', operation=name, counts=counts.copy())
        raise AssertionError('Unregistered operation: ' + name)

    return reject


def monitored(original, counter, returned=None, capture=None):
    @wraps(original)
    def call(*args, **kwargs):
        consume(counter)
        event('started', operation=counter, counts=counts.copy())
        try:
            value = original(*args, **kwargs)
        except BaseException as error:
            event(
                'raised',
                operation=counter,
                error_type=type(error).__name__,
                reason=str(error),
                counts=counts.copy(),
            )
            raise
        if returned is not None:
            counts[returned] += 1
        if capture is not None:
            capture(value, counts[returned] if returned else counts[counter])
        if returned is not None:
            assert counts[returned] <= budgets[returned], 'Unexpected return preserved'
        event('returned', operation=counter, counts=counts.copy())
        return value

    return call


def retain_preparation(value, number):
    preparations.append(value)
    save(output / f'preparation_{number}_declaration.json', value.to_dict())
    save(output / f'preparation_{number}_inspection.json', inspect_prepared(value))


def retain_reconstruction(value, number):
    record = inspect_native(value)
    reconstructions.append((value, record))
    save(output / f'reconstruction_{number}.json', record)


def retain_score(value, number):
    returned_scores.append(value)
    save(output / f'returned_score_{number}.json', value.to_dict())


def install_monitoring():
    patch(
        dmt,
        'prepare_ligand',
        monitored(dmt.prepare_ligand, 'prepare', capture=retain_preparation),
    )
    patch(
        dmt.DockingPose,
        'to_molecular_system',
        monitored(
            dmt.DockingPose.to_molecular_system,
            'reconstruct',
            capture=retain_reconstruction,
        ),
    )
    patch(
        dmt,
        'score',
        monitored(dmt.score, 'score_attempts', 'score_returns', retain_score),
    )
    patch(
        dmt,
        'compare_scores',
        monitored(
            dmt.compare_scores,
            'comparison_attempts',
            'comparison_returns',
            lambda value, n: save(output / f'returned_comparison_{n}.json', value),
        ),
    )
    patch(
        msm.build,
        'assign_partial_charges',
        monitored(msm.build.assign_partial_charges, 'source_charge_assignment'),
    )
    patch(
        msm.build,
        'assign_autodock_atom_types',
        monitored(msm.build.assign_autodock_atom_types, 'source_type_assignment'),
    )
    patch(
        msm.chemical_states,
        'get_atom_correspondences',
        monitored(
            msm.chemical_states.get_atom_correspondences,
            'graph_witness',
            capture=lambda value, n: save(output / f'graph_witness_{n}.json', value),
        ),
    )
    original_audit = dmt.audit_preparation_charges
    audit = monitored(
        original_audit,
        'charge_audit',
        capture=lambda value, n: save(output / f'charge_audit_{n}.json', value),
    )
    patch(dmt, 'audit_preparation_charges', audit)
    patch(comparison_api, 'audit_preparation_charges', audit)
    patch(
        vina.Vina, '__init__', monitored(vina.Vina.__init__, 'native_initializations')
    )
    patch(
        vina.Vina,
        'compute_vina_maps',
        monitored(vina.Vina.compute_vina_maps, 'native_maps'),
    )
    patch(
        vina.Vina,
        'score',
        monitored(
            vina.Vina.score,
            'native_scores',
            capture=lambda value, n: save(output / f'native_score_{n}.json', value),
        ),
    )
    for owner, name in [
        (dmt, 'dock'),
        (dmt, 'prepare_receptor'),
        (msm.build, 'add_missing_hydrogens'),
        (vina.Vina, 'dock'),
        (vina.Vina, 'optimize'),
        (vina.Vina, 'randomize'),
        (vina.Vina, 'energies'),
    ]:
        patch(owner, name, forbidden(name))


def witness(system, reference, label):
    left = inspect_native(system, mechanics=False)
    right = inspect_native(reference, mechanics=False)
    for field in ('ids', 'names', 'elements'):
        assert left[field] == right[field], ('Unadmitted axis', field)
    candidate = [[i, i] for i in range(len(left['ids']))]
    report = normalized(
        msm.chemical_states.get_atom_correspondences(
            system,
            reference_molecular_system=reference,
            structure_indices=0,
            reference_structure_indices=0,
            chemical_state='structure',
            reference_chemical_state='structure',
            candidate_atom_correspondences=[candidate],
            max_matches=1,
        )
    )
    save(output / (label + '.json'), report)
    assert report['schema'] == 'molsysmt.atom_correspondences@1'
    assert report['search_scope'] == 'supplied_atom_correspondences'
    assert report['search_complete'] is True
    assert report['evidence']['kind'] == 'declared_complete_covalent_graph'
    assert report['atom_correspondences'] == [candidate]
    return report


def require_standalone_limits(record, label):
    audit = dmt.audit_pose(record)
    save(output / (label + '_standalone_audit.json'), audit)
    incomplete = [row for row in audit['checks'] if row['status'] == 'incomplete']
    assert audit['status'] == 'incomplete'
    assert not any(row['status'] == 'inconsistent' for row in audit['checks'])
    assert len(incomplete) == 4
    assert {row['path'] for row in incomplete} == {
        '/scores/' + name for name in ('vina', 'inter', 'intra', 'torsion')
    }
    assert all(
        row['check'] == 'score_origin'
        and row['reason'] == 'evaluation_evidence_missing'
        for row in incomplete
    )

## Ordinary public workflow and its unit cost

The maintained three-cell example receives ordinary kernel objects: `source_result`,
`source_prepared`, `source_receptor`, `source_domain`, `active_torsion_selections=[]`
and a fresh `output_directory`. It uses no qualification arguments or helper API.
This receiving executes those exact unmodified cells in their own namespace.

For every actual original pose it reconstructs once, prepares once without
H/charge/type calculation, and adds two named fixed evaluations (sixteen values).
Here the complete source is reproduced once beforehand with one explicit charge
assignment and one explicit type assignment. Both extra score attempts below
must refuse before native initialization; the two comparison calls recalculate
no energy. All N=2..3 returned original poses remain in the population.


In [ ]:
def receive_preparation():
    full18 = msm.convert(source_path, to_form='molsysmt.MolSys')
    full18_before = inspect_native(full18, mechanics=False)
    assert len(full18_before['ids']) == 18
    charged = msm.build.assign_partial_charges(full18, method='gasteiger_marsili')
    typed = msm.build.assign_autodock_atom_types(
        charged, typing_scheme='autodock4', method='chemical_environment'
    )
    typed_before = inspect_native(typed)
    save(
        output / 'fresh_typed18_inspection.json',
        dict(
            native=typed_before,
            scope='New named assignments; inspection only, never an MM restoration input',
        ),
    )
    source_prepared = dmt.prepare_ligand(
        typed,
        selection='all',
        state_id=source_result.poses[0].partner_state_id,
        active_torsion_bonds=[],
    )
    source_before = inspect_prepared(source_prepared)
    assert source_prepared.n_atoms == 13 and source_prepared.torsion_dof == 0
    assert source_prepared.pdbqt_atom_indices == list(range(13))
    prior = generated_inspection
    for field in (
        'state_id',
        'n_atoms',
        'atom_names',
        'atom_types',
        'group_names',
        'group_ids',
        'torsion_dof',
        'pdbqt_atom_indices',
    ):
        assert source_before['declaration'][field] == prior['declaration'][field], field
    for field in ('ids', 'names', 'elements', 'types'):
        assert source_before['native'][field] == prior['native'][field], field
    for field in ('charges_e',):
        np.testing.assert_allclose(
            source_before['native'][field], prior['native'][field], rtol=0, atol=1e-12
        )
    np.testing.assert_allclose(
        source_before['coordinates_nm'], prior['coordinates_nm'], rtol=0, atol=1e-12
    )
    np.testing.assert_allclose(
        source_prepared.charges, prior['declaration']['charges'], rtol=0, atol=1e-12
    )
    source_text = source_prepared.to_pdbqt()
    (output / 'reproduced_source_ligand.pdbqt').write_text(source_text)
    artifacts = source_record['provenance']['backend_artifacts']
    assert (
        hashlib.sha256(source_text.encode()).hexdigest()
        == artifacts['partner']['sha256']
    )
    assert source_text.encode() == base64.b64decode(
        artifacts['partner']['content_base64'], validate=True
    )
    assert receptor_path.read_bytes() == base64.b64decode(
        artifacts['receptor']['content_base64'], validate=True
    )
    witness(
        source_prepared.source_molsys,
        source_prepared.source_molsys,
        'reproduced_source13_witness',
    )
    source_charge_audit = dmt.audit_preparation_charges(source_prepared)
    save(output / 'source_charge_audit.json', source_charge_audit)
    assert source_charge_audit['assessment'] == 'consistent'
    source_domain = dmt.BoxRegion(
        puw.quantity(contract['backend_box']['center'], 'angstrom'),
        puw.quantity([24.0, 24.0, 24.0], 'angstrom'),
    )
    save(
        output / 'source_reproduction_gate.json',
        dict(
            generation_manifest_sha256=generation_sha,
            source_result_wire_sha256=hashlib.sha256(source_wire_bytes).hexdigest(),
            reproduced_partner_sha256=artifacts['partner']['sha256'],
            receptor_sha256=artifacts['receptor']['sha256'],
            source_population_size=population_size,
            source_docking_box=source_record['provenance']['backend_box'],
            new_scoring_box=source_domain.to_backend_box(unit='angstrom'),
            original_scores_ranks_history_preserved=True,
            generated_inspection_compared_only=True,
            historical_preparation_restored=False,
        ),
    )
    assert counts['prepare'] == counts['source_charge_assignment'] == 1
    assert counts['source_type_assignment'] == 1
    assert counts['native_initializations'] == counts['native_scores'] == 0
    return SimpleNamespace(
        full18=full18,
        full18_before=full18_before,
        typed=typed,
        typed_before=typed_before,
        source_prepared=source_prepared,
        source_before=source_before,
        source_domain=source_domain,
        source_domain_before=deepcopy(source_domain.to_dict()),
    )


def receive_public_example(state):
    namespace = dict(
        __name__='__main__',
        source_result=source_result,
        source_prepared=state.source_prepared,
        source_receptor=receptor_path,
        source_domain=state.source_domain,
        active_torsion_selections=[],
        output_directory=output / 'user',
    )
    for cell in user_cells:
        text = ''.join(cell['source'])
        event(
            'portable-cell-started',
            cell=cell['id'],
            sha256=hashlib.sha256(text.encode()).hexdigest(),
            counts=counts.copy(),
        )
        exec(compile(text, 'public_rescore_example.' + cell['id'], 'exec'), namespace)
        event('portable-cell-completed', cell=cell['id'], counts=counts.copy())
    state.namespace = namespace
    state.updated = namespace['rescored_result']
    state.problems = namespace['prepared_problems']
    assert len(state.updated) == len(state.problems) == population_size
    assert counts['prepare'] == population_size + 1
    assert counts['reconstruct'] == population_size
    assert counts['score_attempts'] == counts['score_returns'] == 2 * population_size
    assert (
        counts['native_initializations'] == counts['native_maps'] == 2 * population_size
    )
    assert counts['native_scores'] == 2 * population_size
    return state


def check_sources(state):
    assert source_result.to_dict() == source_record
    assert inspect_prepared(state.source_prepared) == state.source_before
    assert inspect_native(state.full18, mechanics=False) == state.full18_before
    assert inspect_native(state.typed) == state.typed_before
    assert state.source_domain.to_dict() == state.source_domain_before
    assert state.namespace['active_torsion_selections'] == []
    for system, before in reconstructions:
        assert inspect_native(system) == before

In [ ]:
def receive_records_and_comparison(state):
    updated = state.updated.to_dict()
    save(output / 'updated_result.json', updated)
    for field in ('problem_info', 'protocol_info', 'provenance'):
        assert updated[field] == source_record[field], field
    assert state.updated.ranking_history == source_result.ranking_history
    assert len(reconstructions) == population_size
    for index, (old, current, problem, reconstructed) in enumerate(
        zip(source_record['poses'], updated['poses'], state.problems, reconstructions),
        1,
    ):
        native, native_before = reconstructed
        prepared = problem.partner
        assert inspect_native(native) == native_before
        assert prepared.state_id == state.source_prepared.state_id
        assert prepared.n_atoms == 13 and prepared.torsion_dof == 0
        assert prepared.pdbqt_atom_indices == list(range(13))
        witness(
            native,
            state.source_prepared.source_molsys,
            f'pose_{index}_reconstructed_witness',
        )
        witness(prepared.source_molsys, native, f'pose_{index}_prepared_witness')
        audit = dmt.audit_preparation_charges(prepared)
        save(output / f'pose_{index}_charge_audit.json', audit)
        assert audit['assessment'] == 'consistent'
        np.testing.assert_allclose(
            prepared.charges, state.source_prepared.charges, rtol=0, atol=1e-12
        )
        assert prepared.atom_types == state.source_prepared.atom_types
        for field in ('ids', 'names', 'elements', 'types'):
            assert native_before[field] == state.source_before['native'][field], field
        np.testing.assert_allclose(
            native_before['charges_e'],
            state.source_before['native']['charges_e'],
            rtol=0,
            atol=1e-12,
        )
        np.testing.assert_allclose(
            puw.get_value(prepared.coordinates, to_unit='nm'),
            native_before['coordinates_nm'][0],
            rtol=0,
            atol=1e-12,
        )
        np.testing.assert_allclose(
            puw.get_value(prepared.coordinates, to_unit='nm'),
            puw.get_value(source_result.poses[index - 1].coordinates, to_unit='nm'),
            rtol=0,
            atol=1e-12,
        )
        for field in (
            'pose_id',
            'rank',
            'partner_state_id',
            'receptor_state_id',
            'coordinates',
        ):
            assert current[field] == old[field], field
        assert len(current['scores']) == 20
        assert all(
            current['scores'][name] == value for name, value in old['scores'].items()
        )
        previous_definitions = old['metadata']['score_definitions']
        assert all(
            current['metadata']['score_definitions'][name] == value
            for name, value in previous_definitions.items()
        )
        assert {
            k: v
            for k, v in current['metadata'].items()
            if k not in ('score_definitions', 'scoring_history')
        } == {
            k: v
            for k, v in old['metadata'].items()
            if k not in ('score_definitions', 'scoring_history')
        }
        history = current['metadata']['scoring_history']
        assert len(history) == 2
        assert [entry['score_name'] for entry in history] == [
            'vina_fixed',
            'vinardo_fixed',
        ]
        for entry in history:
            assert len(entry['scores']) == len(entry['score_definitions']) == 8
            dmt.verify_captured_inputs(entry['backend_artifacts'])
            assert (
                entry['backend_artifacts']['receptor']['sha256']
                == contract['receptor_sha256']
            )
            assert entry['backend_box'] == {
                'center': contract['backend_box']['center'],
                'size': [24.0, 24.0, 24.0],
                'unit': 'angstrom',
            }
        assert json.loads(json.dumps(current, allow_nan=False)) == current
        require_standalone_limits(current, f'pose_{index}')
    for index, returned in enumerate(returned_scores, 1):
        require_standalone_limits(returned.to_dict(), f'returned_score_{index}')
    audit = dmt.audit_result(updated)
    save(output / 'combined_audit.json', audit)
    assert audit['status'] == 'consistent'
    assert dmt.DockingResult.from_dict(deepcopy(updated)).to_dict() == updated
    check_sources(state)
    prepared_before = [inspect_prepared(problem.partner) for problem in state.problems]
    problem_before = [deepcopy(problem.to_dict()) for problem in state.problems]
    comparisons = {}
    comparison_directory = output / 'comparisons'
    comparison_directory.mkdir()
    for label in ('vina_fixed', 'vinardo_fixed'):
        report = dmt.compare_scores(
            state.updated,
            label + '.lig_inter',
            problems=state.problems,
            ascending=True,
        )
        save(comparison_directory / (label + '.json'), report)
        pristine = deepcopy(report)
        assert len(report['rows']) == population_size
        assert [row['position'] for row in report['rows']] == list(
            range(1, population_size + 1)
        )
        assert sorted(report['display_order']) == list(range(1, population_size + 1))
        assert [row['pose_id'] for row in report['rows']] == [
            pose.pose_id for pose in source_result
        ]
        assert [row['original_rank'] for row in report['rows']] == [
            pose.rank for pose in source_result
        ]
        assert [row['value'] for row in report['rows']] == [
            pose.scores[label + '.lig_inter'] for pose in state.updated
        ]
        assert all(
            row['evaluation_audit']['status'] == 'consistent' for row in report['rows']
        )
        assert (
            len(
                {
                    row['definition']['context']['partner_sha256']
                    for row in report['rows']
                }
            )
            == population_size
        )
        assert json.loads(json.dumps(report, allow_nan=False)) == pristine
        # Mutate the actual return after saving it; no scientific model changes.
        report['rows'][0]['definition']['context']['weights'][0] = 999
        report['rows'][0]['prepared_evidence']['export']['source_atom_ids'][0] = (
            'detachment-control'
        )
        assert state.updated.to_dict() == updated
        assert [
            inspect_prepared(problem.partner) for problem in state.problems
        ] == prepared_before
        report.clear()
        report.update(pristine)
        comparisons[label] = report
    assert [problem.to_dict() for problem in state.problems] == problem_before
    assert [
        inspect_prepared(problem.partner) for problem in state.problems
    ] == prepared_before
    assert counts['graph_witness'] == 4 * population_size + 1
    assert counts['charge_audit'] == 3 * population_size + 1
    state.updated_record = updated
    state.prepared_before = prepared_before
    state.problem_before = problem_before
    state.comparisons = comparisons
    return state


def receive_refusals(state):
    # Select an informative control pair without dropping any original pose.
    geometry_pair = None
    for first in range(population_size):
        for second in range(first + 1, population_size):
            left = puw.get_value(
                source_result.poses[first].coordinates, to_unit='angstrom'
            )
            right = puw.get_value(
                source_result.poses[second].coordinates, to_unit='angstrom'
            )
            if not np.allclose(left, right, rtol=0, atol=2 * 0.000500001):
                geometry_pair = (first, second)
                break
        if geometry_pair is not None:
            break
    assert geometry_pair is not None, 'No informative geometry-swap control; stop.'
    first, second = geometry_pair
    save(
        output / 'geometry_control_admission.json',
        dict(
            original_positions=[first + 1, second + 1],
            coordinate_tolerance=dict(value=0.000500001, unit='angstrom'),
            coordinates_differ_beyond_twice_tolerance=True,
            all_population_members_retained=True,
        ),
    )
    changed_record = deepcopy(source_record['poses'][0])
    changed_record['partner_state_id'] = 'changed-control-state'
    changed_pose = dmt.DockingPose.from_dict(changed_record)
    controls = [
        (
            'changed_state',
            state.problems[0],
            changed_pose,
            'partner state does not match',
        ),
        (
            'wrong_prepared_geometry',
            state.problems[second],
            source_result.poses[first],
            'Pose coordinates must match',
        ),
    ]
    refusals = []
    native_keys = ('native_initializations', 'native_maps', 'native_scores')
    for label, problem, pose, expected in controls:
        before = pose.to_dict()
        native_before = {key: counts[key] for key in native_keys}
        try:
            dmt.score(
                problem,
                dmt.VinaProtocol(
                    scoring='vina',
                    cpu=1,
                    seed=7,
                    capture_backend_inputs=True,
                    collect_timings=True,
                ),
                pose=pose,
                score_name='control_' + label,
            )
        except CatalogException as error:
            refused = dict(
                control=label,
                code=error.code,
                reason=str(error),
                native_counts_before=native_before,
                native_counts_after={key: counts[key] for key in native_keys},
            )
            save(output / (label + '_refusal.json'), refused)
            assert expected in str(error), str(error)
            assert refused['native_counts_before'] == refused['native_counts_after']
            consume('expected_score_refusals')
            refusals.append(refused)
        else:
            raise AssertionError('Unexpected native scoring return: ' + label)
        assert pose.to_dict() == before
    for label in ('vina_fixed', 'vinardo_fixed'):
        before = state.updated.to_dict()
        try:
            unexpected = state.updated.rank_by(label + '.lig_inter')
        except CatalogException as error:
            record = dict(
                control='strict_rank_' + label, code=error.code, reason=str(error)
            )
            save(output / ('strict_rank_' + label + '_refusal.json'), record)
            assert 'context' in str(error).lower(), str(error)
            consume('rank_refusals')
            refusals.append(record)
        else:
            save(output / ('unexpected_rank_' + label + '.json'), unexpected.to_dict())
            raise AssertionError(
                'Strict rank_by unexpectedly ordered distinct input hashes'
            )
        assert state.updated.to_dict() == before
    save(output / 'refusals.json', refusals)
    assert state.updated.to_dict() == state.updated_record
    assert [problem.to_dict() for problem in state.problems] == state.problem_before
    assert [
        inspect_prepared(problem.partner) for problem in state.problems
    ] == state.prepared_before
    check_sources(state)
    return state

In [ ]:
caller_policy = puw.configure.report()
units = [
    'pm',
    'fs',
    'kelvin',
    'mole',
    'dalton',
    'coulomb',
    'kJ/mol',
    'kJ/(mol*pm)',
    'kJ/(mol*pm**2)',
    'radians',
]
save(
    output / 'registered_operation_budgets.json',
    dict(
        owner='uibcdf/bindingmt#78',
        actual_population=population_size,
        budgets=budgets,
        caller_policy=caller_policy,
        native_policy='pm/fs/coulomb; one native receiving, no second-policy scoring',
    ),
)
state = None
failure = None
stage = 'monitoring'
try:
    with puw.context(standard_units=units):
        active_policy = puw.configure.report()
        install_monitoring()
        stage = 'source-preparation-gate'
        state = receive_preparation()
        assert puw.configure.report() == active_policy
        stage = 'unchanged-public-example'
        state = receive_public_example(state)
        assert puw.configure.report() == active_policy
        stage = 'molecular-record-comparison-evidence'
        state = receive_records_and_comparison(state)
        assert puw.configure.report() == active_policy
        stage = 'pre-native-and-strict-ranking-refusals'
        state = receive_refusals(state)
        assert puw.configure.report() == active_policy
        assert counts == budgets, (counts, budgets)
        assert not forbidden_attempts
        for path, expected in stored_inputs.items():
            assert file_sha(Path(path)) == expected
        for path, expected in local_input_hashes.items():
            assert file_sha(Path(path)) == expected
        save(
            output / 'scientific_observations.json',
            dict(
                actual_source_poses=population_size,
                original_score_values=4 * population_size,
                new_fixed_score_values=16 * population_size,
                all_original_poses_retained=True,
                original_result_ranks_scores_metadata_and_history_preserved=True,
                collection_audit='consistent',
                standalone_audits='incomplete only for the four collection-owned original scores',
                current_prepared_views_and_graph_witnesses_received=True,
                current_source_assignments=dict(charges=1, types=1),
                public_comparison_reports=2,
                comparison_orders={
                    name: report['display_order']
                    for name, report in state.comparisons.items()
                },
                scope='New rigid docking population, current source reproduction and public additive continuity; no historical H restoration or energy accuracy claim',
            ),
        )
except BaseException as error:
    failure = error
    save(
        output / 'failure.json',
        dict(
            stage=stage,
            error_type=type(error).__name__,
            reason=str(error),
            counts=counts.copy(),
            budgets=budgets,
            policy='Preserve partial returns and stop; no automatic retry or subset PASS',
        ),
    )
finally:
    for owner, name, original in reversed(patches):
        setattr(owner, name, original)
    save(
        output / 'operation_outcome.json',
        dict(
            stage=stage,
            counts=counts.copy(),
            budgets=budgets,
            scientific_steps_completed=failure is None,
            forbidden_attempts=forbidden_attempts,
        ),
    )
try:
    assert puw.configure.report() == caller_policy
    installed_check('after-continuity-receiving')
except BaseException as cleanup_error:
    save(
        output / 'final_integrity_failure.json',
        dict(
            error_type=type(cleanup_error).__name__,
            reason=str(cleanup_error),
            earlier_failure=type(failure).__name__ if failure is not None else None,
        ),
    )
    if failure is None:
        failure = cleanup_error
    else:
        failure.add_note('Final integrity check also failed: ' + str(cleanup_error))
if failure is not None:
    raise failure
save(
    output / 'receipt.json',
    dict(
        schema='bindingmt.e1_rigid_continuity_receiving@1',
        owner='uibcdf/bindingmt#78',
        outcome='PASS',
        generation_manifest_sha256=generation_sha,
        construction_manifest_sha256=construction_sha,
        user_notebook_sha256=user_notebook_sha,
        counts=counts,
        actual_poses=population_size,
        native_unit_policy='pm/fs/coulomb',
        native_policy_repetitions=1,
        original_values_per_pose=4,
        added_values_per_pose=16,
        successful_fixed_score_calls=2 * population_size,
        public_comparison_returns=2,
        expected_pre_native_score_refusals=2,
        strict_component_ranking_refusals=2,
        original_scores_ranks_and_backend_history_preserved=True,
        generated_sidecars_compared_only=True,
        new_source_charge_assignments=1,
        new_source_type_assignments=1,
        new_hydrogens=0,
        new_searches=0,
        original_incomplete_two_pose_population_reclassified=False,
        historical_prepared13_identity_received=False,
        full_e1_accepted=False,
        scope='Bounded private installed public rigid multi-pose continuity and separate method-local interaction reports; no affinity/accuracy or public platform/dependency closure',
    ),
)
print(
    json.dumps(
        dict(
            outcome='PASS',
            actual_poses=population_size,
            original_values=4 * population_size,
            added_values=16 * population_size,
            comparison_reports=2,
            counts=counts,
        )
    )
)